# 🔮 Notebook 04 — Temporal Prediction
## ParkSense AI | Gridlock Round 2

**Input:** `data/processed/clustered_violations.parquet` & `hotspots.parquet`  
**Output:** `data/processed/patrol_recommendations.parquet`

This notebook uses XGBoost to predict the volume and severity of congestion for the upcoming week using rigorous Time-Series Cross Validation and Poisson counting.

In [6]:
import pandas as pd
import numpy as np
import warnings
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error
import xgboost as xgb

warnings.filterwarnings('ignore')
print('✅ Imports done.')

✅ Imports done.


In [7]:
DATA_PATH = '../data/processed/clustered_violations.parquet'
HOTSPOTS_PATH = '../data/processed/hotspots.parquet'

df = pd.read_parquet(DATA_PATH)
hotspots_df = pd.read_parquet(HOTSPOTS_PATH)

df_hotspots = df[df['hotspot_rank'] != -1].copy()
df_hotspots['created_datetime'] = pd.to_datetime(df_hotspots['created_datetime'])
df_hotspots['date'] = df_hotspots['created_datetime'].dt.date
df_hotspots['hour'] = df_hotspots['created_datetime'].dt.hour

# Aggregate by Exact Date
actual_counts = df_hotspots.groupby(['hotspot_rank', 'date', 'hour']).agg(
    target_violation_count=('id', 'count'),
    target_avg_pici=('pici_score', 'mean')
).reset_index()

# Build Complete Grid
min_date = df_hotspots['date'].min()
max_date = df_hotspots['date'].max()
date_range = pd.date_range(start=min_date, end=max_date, freq='D').date
hours = list(range(24))
hotspot_ranks = hotspots_df['hotspot_rank'].unique()

grid = [[h, d, hr] for h in hotspot_ranks for d in date_range for hr in hours]
grid_df = pd.DataFrame(grid, columns=['hotspot_rank', 'date', 'hour'])

master_df = pd.merge(grid_df, actual_counts, on=['hotspot_rank', 'date', 'hour'], how='left')
master_df['target_violation_count'] = master_df['target_violation_count'].fillna(0)
master_df['target_avg_pici'] = master_df['target_avg_pici'].fillna(0)
print(f'✅ Complete timeline built. {len(master_df):,} rows.')

✅ Complete timeline built. 714,024 rows.


In [8]:
master_df['date_dt'] = pd.to_datetime(master_df['date'])
master_df['day_of_week'] = master_df['date_dt'].dt.dayofweek
master_df['month'] = master_df['date_dt'].dt.month
master_df['is_weekend'] = master_df['day_of_week'].apply(lambda d: 1 if d >= 5 else 0)

HOLIDAYS = [
    '2023-11-12', '2023-11-13', '2023-12-25', '2024-01-01', '2024-01-15',
    '2024-01-26', '2024-03-08', '2024-03-25', '2024-04-09', '2024-04-11', '2024-05-01'
]
holidays_dt = pd.to_datetime(HOLIDAYS).date
master_df['is_holiday'] = master_df['date'].isin(holidays_dt).astype(int)

master_df['is_peak_hour'] = master_df.apply(
    lambda r: 1 if (r['is_weekend'] == 0 and r['is_holiday'] == 0) and ((8 <= r['hour'] <= 11) or (17 <= r['hour'] <= 21)) else 0,
    axis=1
)
master_df['is_business_hours'] = master_df.apply(
    lambda r: 1 if (r['is_weekend'] == 0 and r['is_holiday'] == 0) and (9 <= r['hour'] < 18) else 0,
    axis=1
)

master_df = pd.merge(master_df, hotspots_df[['hotspot_rank', 'center_lat', 'center_lng']], on='hotspot_rank', how='left')
master_df = master_df.sort_values(['date', 'hour', 'hotspot_rank']).reset_index(drop=True)

FEATURES = ['center_lat', 'center_lng', 'day_of_week', 'hour', 'month', 'is_peak_hour', 'is_business_hours', 'is_holiday']
X = master_df[FEATURES]
y_volume = master_df['target_violation_count']
y_pici = master_df['target_avg_pici']

In [ ]:
# Date-based Split
unique_dates = master_df['date'].unique()
split_date_idx = int(len(unique_dates) * 0.8)
split_date = unique_dates[split_date_idx]

train_mask = master_df['date'] < split_date
test_mask = master_df['date'] >= split_date

X_train, X_test = X[train_mask], X[test_mask]
y_vol_train, y_vol_test = y_volume[train_mask], y_volume[test_mask]
y_pic_train, y_pic_test = y_pici[train_mask], y_pici[test_mask]

tscv = TimeSeriesSplit(n_splits=3)
param_grid = {'max_depth': [3, 5, 7], 'learning_rate': [0.05, 0.1], 'n_estimators': [100, 200]}

print("Training Volume Model with count:poisson...")
grid_vol = GridSearchCV(estimator=xgb.XGBRegressor(objective='count:poisson', subsample=0.8, random_state=42), param_grid=param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_vol.fit(X_train, y_vol_train)
model_volume = grid_vol.best_estimator_

print("Training PICI Model...")
grid_pici = GridSearchCV(estimator=xgb.XGBRegressor(objective='reg:squarederror', subsample=0.8, random_state=42), param_grid=param_grid, cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_pici.fit(X_train, y_pic_train)
model_pici = grid_pici.best_estimator_

vol_preds = model_volume.predict(X_test)
pici_preds = model_pici.predict(X_test).clip(min=0, max=10)

print("\n=== Overall MAE ===")
print(f"Volume MAE: {mean_absolute_error(y_vol_test, vol_preds):.3f}")
print(f"PICI MAE:   {mean_absolute_error(y_pic_test, pici_preds):.3f}")

# Evaluate Active Hours
active_idx = (y_vol_test > 0) | (vol_preds > 0.5)
if active_idx.sum() > 0:
    print("\n=== Active Hours MAE (Where Violations > 0 OR Predicted > 0) ===")
    print(f"Volume MAE: {mean_absolute_error(y_vol_test[active_idx], vol_preds[active_idx]):.3f}")
    print(f"PICI MAE:   {mean_absolute_error(y_pic_test[active_idx], pici_preds[active_idx]):.3f}")

Training Volume Model with count:poisson...
Training PICI Model...

=== Overall MAE ===
Volume MAE: 0.045
PICI MAE:   0.005

=== Active Hours MAE (Where Violations > 0 OR Predicted > 0) ===
Volume MAE: 3.335
PICI MAE:   0.309


In [10]:
print("Generating Patrol Recommendations...")
future_grid = []
for idx, row in hotspots_df.iterrows():
    h_rank = row['hotspot_rank']
    c_lat, c_lng = row['center_lat'], row['center_lng']
    for d in range(7):
        for hr in range(24):
            is_wknd = 1 if d >= 5 else 0
            is_peak = 1 if (is_wknd == 0) and ((8 <= hr <= 11) or (17 <= hr <= 21)) else 0
            is_bus = 1 if (is_wknd == 0) and (9 <= hr < 18) else 0
            future_grid.append({
                'hotspot_rank': h_rank, 'center_lat': c_lat, 'center_lng': c_lng,
                'day_of_week': d, 'hour': hr, 'month': max_date.month, 
                'is_peak_hour': is_peak, 'is_business_hours': is_bus, 'is_holiday': 0
            })

schedule_df = pd.DataFrame(future_grid)
schedule_df['predicted_violations'] = model_volume.predict(schedule_df[FEATURES]).clip(min=0)
schedule_df['predicted_pici'] = model_pici.predict(schedule_df[FEATURES]).clip(min=0, max=10)

# Logical Override
schedule_df.loc[schedule_df['predicted_violations'] < 0.1, 'predicted_pici'] = 0.0
schedule_df['priority_score'] = schedule_df['predicted_violations'] * schedule_df['predicted_pici']

schedule_df = schedule_df.sort_values('priority_score', ascending=False).reset_index(drop=True)

OUT_PATH = '../data/processed/patrol_recommendations.parquet'
schedule_df.to_parquet(OUT_PATH, index=False)
print('✅ Patrol recommendations saved.')

Generating Patrol Recommendations...
✅ Patrol recommendations saved.
